# Gasificador 0D — Señales de BC: rampas, escalones, pulsos y control

**Objetivo:** demostrar que `T_wall`, `Qwall` y otras BC del gasificador admiten
señales variables — `callable(t)` para perfiles temporales y `callable(t, snap)` para
retroalimentación de estado — sin cambiar nada en el RHS ni en el runner.

Todas las simulaciones son **0D, batch** (N=1, v_out=0.0, sin flujo externo de gas).
El caso de referencia es T_wall constante a 1073.15 K (800 °C), idéntico al test_01.

| Caso | BC modificada | Tipo de señal | Señal |
|------|--------------|---------------|-------|
| **Ref** | T_wall = 1073.15 K | constante | — |
| **3A** | T_wall | `ramp(t)` | 300 → 1073 K a 0.214 K/s |
| **3B** | T_wall | `step(t)` | 500 K → 1073 K a t=600 s |
| **3C** | Qwall | `pulse(t)` | 500 W en t∈[0, 600 s], 0 W después |
| **3D** | T_wall | `piecewise(t)` | 4 etapas: calentar → plateau → calentar → operar |
| **3E** | T_wall | `proportional(t, snap)` | control P → mantiene Ts_mean ≈ 700 K |

**API del runner** (igual que test_01 y test_02):  
`t_arr, _, gasifier = run_step(sv0, t_max, params, rtol, atol, n_sec, show_progress)`

**Atributos del objeto gasifier:**  
`_Ts_results (n_t,N)`, `_Tg_results (n_t,N)`, `_P_results (n_t,N)`,  
`_rho_solid_results (n_t,3,N)` — [0]=biomasa, [1]=char, [2]=humedad,  
`_y_results (n_t,nc,N)`, `_v_out_results (n_t,)`

**Referencias:**
- Señales: `src.control.signals` — ramp, step, pulse, piecewise
- Controladores: `src.control.controllers` — proportional
- Arquitectura: `.claude/equipment/signal-bc-integration.md`

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.io.fuels_reader                      import read_fueldb
from src.units.gasifier.config.boundary_c     import build_bc_config
from src.units.gasifier.config.thermal_bc     import build_thermal_bc_config
from src.units.gasifier.config.transport      import build_transport_config
from src.units.gasifier.config.gas_props      import build_gas_prop_config, GASIFIER_GAS_SPECIES
from src.units.gasifier.config.solid_props    import build_solid_prop_config
from src.units.gasifier.config.initial_c      import build_initial_c_config
from src.solvers.runner_gasifier              import run_step
from src.postprocessing.gasifier_balances     import check_balances
from src.control.signals                      import ramp, step, pulse, piecewise
from src.control.controllers                  import proportional
from src.utils.signals                        import resolve

FUEL_PATH = os.path.join(ROOT, "materials", "fuels", "softwood_spruce.yaml")
GAS_DB    = os.path.join(ROOT, "materials", "fluids", "gasdb.txt")

print("Imports OK")

In [ ]:
# ── Combustible y geometría ───────────────────────────────────────────────────
fuel_config = read_fueldb(FUEL_PATH)
nc          = 9
N           = 1              # 0D — reactor perfectamente mezclado
species     = list(GASIFIER_GAS_SPECIES)

Di, Do  = 0.10, 0.114         # [m] diámetros interno/externo
e_wall  = (Do - Di) / 2       # [m] espesor de pared (7 mm, acero inox)
L       = 0.50                # [m] longitud
dz      = L / N
Ai      = 0.25 * np.pi * Di**2
Pi, Po  = np.pi * Di, np.pi * Do
epsi_r  = 0.60
dp0     = float(fuel_config["physical"]["dp_initial"])
rho_p   = float(fuel_config["physical"]["rho_particle"])
rho_char0 = fuel_config["pyrolysis_yields"]["char"] * rho_p * (1 - epsi_r)

# ── Condiciones de operación comunes ─────────────────────────────────────────
MC_WB   = 0.165    # [-]  contenido de humedad en base húmeda (16.5 %)
P_OUT   = 1.01325  # [bar]
T_END   = 3600.0   # [s]  duración
RTOL    = 1e-5
ATOL    = 1e-7
N_SEC   = 3        # puntos/s de salida (3 × 3600 s ≈ 10 800 puntos)

# ── Propiedades comunes ───────────────────────────────────────────────────────
prop_gas    = build_gas_prop_config(fuel_config=fuel_config, mode="polynomial", db_path=GAS_DB)
solid_cfg   = build_solid_prop_config(fuel_config)
gas_T_ref   = float(np.min(np.asarray(prop_gas["Tref"])))
MW_arr      = np.asarray(prop_gas["MW"])
trans_cfg   = build_transport_config(mode="constant", N=N, n_comp=nc, h_bed=80.0, h_wall=12.0)
bc_batch    = build_bc_config(n_comp=nc, P_out_bar=P_OUT, v_out=0.0)

# ── Condiciones iniciales comunes ─────────────────────────────────────────────
rho_bio_0 = rho_p * (1 - epsi_r)              # [kg/m³_bed]
rho_moi_0 = MC_WB / (1 - MC_WB) * rho_bio_0   # [kg/m³_bed]
y0 = np.zeros(nc); y0[species.index("N2")] = 1.0

init = build_initial_c_config(
    P_init=P_OUT, Tg_init=300.0, Ts_init=300.0, y_init=y0,
    rho_biomass_init=rho_bio_0, rho_char_init=1e-6, rho_moisture_init=rho_moi_0,
    n_comp=nc, N=N, prop_gas=prop_gas, epsi_r=epsi_r, gas_T_ref=gas_T_ref,
)
sv0 = init["sv0"]

# ── Parámetros base (se sobreescribe thermal_bc_config por caso) ──────────────
params_base = {
    "n_comp":nc, "N":N, "dz":dz, "Ai":Ai, "Di":Di, "Pi":Pi, "Po":Po,
    "prop_gas":prop_gas, "MW":MW_arr, "gas_T_ref":gas_T_ref,
    "bc_config":bc_batch, "trans_config":trans_cfg,
    "energy":True, "epsi_r":epsi_r, "dp0":dp0, "rho_char0":rho_char0,
    "fuel_config":fuel_config, "solid_config":solid_cfg, "species":species,
}

def make_params(tbc):
    """Devuelve params completo con el thermal_bc dado y caché limpio."""
    return {**params_base, "thermal_bc_config": tbc, "_cache": {}}

print(f"sv0.shape = {sv0.shape}  (17×N = {17*N})")
print(f"ρ_bio_0={rho_bio_0:.1f} kg/m³   ρ_moi_0={rho_moi_0:.1f} kg/m³")
print(f"T_end={T_END:.0f} s | rtol={RTOL} | atol={ATOL} | n_sec={N_SEC}")

In [ ]:
# ── Caso referencia: T_wall = 1073.15 K constante ────────────────────────────
tbc_ref = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=1073.15,
    k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
)
print("Simulando referencia (T_wall = 1073.15 K constante)...")
t_ref, _, g_ref = run_step(
    sv0=sv0, t_max=T_END, params=make_params(tbc_ref),
    rtol=RTOL, atol=ATOL, n_sec=N_SEC, show_progress=True,
)
print(f"✓  t={t_ref[-1]:.0f} s | Ts={g_ref._Ts_results[-1,0]-273.15:.0f} °C "
      f"| ρ_bio={g_ref._rho_solid_results[-1,0,0]:.2f} kg/m³")

## 3A — Rampa de T_wall: 300 K → 1073 K en 3600 s

La pared parte de temperatura ambiente y sube linealmente hasta 800 °C a lo largo de toda
la simulación (`slope = (1073.15−300)/3600 ≈ 0.214 K/s`). La señal satura en 1073.15 K.

**Qué observar:** La pirólisis se retrasa respecto al caso constante — la pared tarda en
activar la cinética. La temperatura final del sólido debería ser similar en ambos casos.

In [ ]:
T_wall_ramp = ramp(
    t_start=0.0, slope=(1073.15 - 300.0) / T_END,
    value_init=300.0, value_max=1073.15,
)
print("Verificación T_wall_ramp(t):")
for tc in [0, 600, 1800, 3600]:
    print(f"  t={tc:4d} s → {resolve(T_wall_ramp, float(tc)):.1f} K")

print("\nSimulando 3A (T_wall rampa 300→1073 K)...")
t_3a, _, g_3a = run_step(
    sv0=sv0, t_max=T_END,
    params=make_params(build_thermal_bc_config(
        mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
        T_wall=T_wall_ramp, k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
    )),
    rtol=RTOL, atol=ATOL, n_sec=N_SEC, show_progress=True,
)
print(f"✓  Ts={g_3a._Ts_results[-1,0]-273.15:.0f} °C | ρ_bio={g_3a._rho_solid_results[-1,0,0]:.2f} kg/m³")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("3A — Rampa de T_wall: 300→1073 K en 3600 s  (0D, batch)", fontsize=11)

axes[0].plot(t_3a/60, [resolve(T_wall_ramp, ti) for ti in t_3a], 'r-')
axes[0].axhline(1073.15, color='k', ls='--', alpha=0.5, label='referencia')
axes[0].set(xlabel="t [min]", ylabel="T_wall [K]", title="Señal T_wall")
axes[0].legend(); axes[0].grid(True)

axes[1].plot(t_3a/60,  g_3a._Ts_results[:,0]-273.15, 'r-',  label='rampa')
axes[1].plot(t_ref/60, g_ref._Ts_results[:,0]-273.15, 'k--', label='referencia')
axes[1].set(xlabel="t [min]", ylabel="Ts [°C]", title="Temperatura sólido")
axes[1].legend(); axes[1].grid(True)

axes[2].plot(t_3a/60,  g_3a._rho_solid_results[:,0,0], 'r-',  label='rampa')
axes[2].plot(t_ref/60, g_ref._rho_solid_results[:,0,0], 'k--', label='referencia')
axes[2].set(xlabel="t [min]", ylabel="ρ_bio [kg/m³_bed]", title="Biomasa restante")
axes[2].legend(); axes[2].grid(True)

axes[3].plot(t_3a/60,  g_3a._P_results[:,0], 'r-',  label='rampa')
axes[3].plot(t_ref/60, g_ref._P_results[:,0], 'k--', label='referencia')
axes[3].set(xlabel="t [min]", ylabel="P [bar]", title="Presión")
axes[3].legend(); axes[3].grid(True)

plt.tight_layout(); plt.show()
print("\n--- Balances 3A ---")
tbc_3a = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=T_wall_ramp, k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
)
check_balances(g_3a, make_params(tbc_3a))

## 3B — Escalón de T_wall: 500 K → 1073 K a t = 600 s

La pared arranca en 500 K (activa el secado, no la pirólisis activa) y salta a 1073 K
a los 10 minutos. Simula un arranque con precalentamiento previo.

**Qué observar:** El secado ocurre en la fase de precalentamiento; la pirólisis
se activa bruscamente con el salto. La presión sube más tarde que en la referencia.

In [ ]:
T_wall_step = step(t_step=600.0, value_before=500.0, value_after=1073.15)

print("Simulando 3B (escalón T_wall: 500→1073 K a t=600 s)...")
t_3b, _, g_3b = run_step(
    sv0=sv0, t_max=T_END,
    params=make_params(build_thermal_bc_config(
        mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
        T_wall=T_wall_step, k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
    )),
    rtol=RTOL, atol=ATOL, n_sec=N_SEC, show_progress=True,
)
print(f"✓  Ts={g_3b._Ts_results[-1,0]-273.15:.0f} °C | ρ_bio={g_3b._rho_solid_results[-1,0,0]:.2f} kg/m³")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("3B — Escalón T_wall: 500→1073 K a t=10 min  (0D, batch)", fontsize=11)

axes[0].plot(t_3b/60, [resolve(T_wall_step, ti) for ti in t_3b], 'b-', lw=2)
axes[0].set(xlabel="t [min]", ylabel="T_wall [K]", title="Señal T_wall"); axes[0].grid(True)

axes[1].plot(t_3b/60,  g_3b._Ts_results[:,0]-273.15, 'b-',  label='escalón')
axes[1].plot(t_ref/60, g_ref._Ts_results[:,0]-273.15, 'k--', label='referencia')
axes[1].set(xlabel="t [min]", ylabel="Ts [°C]", title="Temperatura sólido")
axes[1].legend(); axes[1].grid(True)

axes[2].plot(t_3b/60,  g_3b._rho_solid_results[:,0,0], 'b-',  label='escalón')
axes[2].plot(t_ref/60, g_ref._rho_solid_results[:,0,0], 'k--', label='referencia')
axes[2].set(xlabel="t [min]", ylabel="ρ_bio [kg/m³_bed]", title="Biomasa restante")
axes[2].legend(); axes[2].grid(True)

axes[3].plot(t_3b/60,  g_3b._P_results[:,0], 'b-',  label='escalón')
axes[3].plot(t_ref/60, g_ref._P_results[:,0], 'k--', label='referencia')
axes[3].set(xlabel="t [min]", ylabel="P [bar]", title="Presión")
axes[3].legend(); axes[3].grid(True)

plt.tight_layout(); plt.show()
print("\n--- Balances 3B ---")
tbc_3b = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=T_wall_step, k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
)
check_balances(g_3b, make_params(tbc_3b))

## 3C — Pulso de Qwall: 500 W durante los primeros 10 min

El modo `heatfluxwall` prescribe la potencia total entregada al reactor. Con
`Qwall=pulse(...)` el calor se inyecta solo en t∈[0, 600 s]; después el sistema es adiabático.

**Qué observar:** ¿Es suficiente la energía inyectada (300 kJ) para pirolizar la carga?
Si la temperatura no supera el umbral cinético, la biomasa queda sin convertir.

In [ ]:
Q_pulse = pulse(t_start=0.0, t_end=600.0, value_on=500.0, value_off=0.0)

print("Simulando 3C (pulso Qwall: 500 W durante 600 s)...")
t_3c, _, g_3c = run_step(
    sv0=sv0, t_max=T_END,
    params=make_params(build_thermal_bc_config(
        mode="heatfluxwall", Di=Di, Do=Do, e_wall=e_wall,
        Qwall=Q_pulse, k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
    )),
    rtol=RTOL, atol=ATOL, n_sec=N_SEC, show_progress=True,
)
print(f"✓  Ts={g_3c._Ts_results[-1,0]-273.15:.0f} °C | ρ_bio={g_3c._rho_solid_results[-1,0,0]:.2f} kg/m³")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("3C — Pulso de Qwall: 500 W en t∈[0, 10 min]  (0D, batch, heatfluxwall)", fontsize=11)

axes[0].plot(t_3c/60, [resolve(Q_pulse, ti) for ti in t_3c], 'g-', lw=2)
axes[0].set(xlabel="t [min]", ylabel="Qwall [W]", title="Señal Qwall"); axes[0].grid(True)

axes[1].plot(t_3c/60, g_3c._Ts_results[:,0]-273.15, 'g-',  label='Ts')
axes[1].plot(t_3c/60, g_3c._Tg_results[:,0]-273.15, 'g--', alpha=0.6, label='Tg')
axes[1].set(xlabel="t [min]", ylabel="T [°C]", title="Temperaturas")
axes[1].legend(); axes[1].grid(True)

axes[2].plot(t_3c/60, g_3c._rho_solid_results[:,0,0], 'g-',  label='biomasa')
axes[2].plot(t_3c/60, g_3c._rho_solid_results[:,1,0], 'g--', alpha=0.7, label='char')
axes[2].set(xlabel="t [min]", ylabel="ρ [kg/m³_bed]", title="Densidades sólidas")
axes[2].legend(); axes[2].grid(True)

plt.tight_layout(); plt.show()
print("\n--- Balances 3C ---")
tbc_3c = build_thermal_bc_config(
    mode="heatfluxwall", Di=Di, Do=Do, e_wall=e_wall,
    Qwall=Q_pulse, k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
)
check_balances(g_3c, make_params(tbc_3c))

## 3D — Piecewise: protocolo de calentamiento por etapas

Perfil de T_wall con 4 etapas definidas mediante `piecewise`:

| Etapa | t [min] | T_wall | Propósito |
|-------|---------|--------|-----------|
| 1 | 0–10 | 300→700 K | Calentamiento inicial / secado |
| 2 | 10–30 | 700 K | Plateau: secado completo |
| 3 | 30–40 | 700→1073 K | Calentamiento hacia pirólisis |
| 4 | 40–60 | 1073 K | Pirólisis a plena temperatura |

In [ ]:
T_wall_pw = piecewise(
    t_breakpoints=[0,    600,   1800,  2400,   3600],
    values=        [300,  700,   700,   1073.15, 1073.15],
)

print("Simulando 3D (piecewise T_wall: 4 etapas)...")
t_3d, _, g_3d = run_step(
    sv0=sv0, t_max=T_END,
    params=make_params(build_thermal_bc_config(
        mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
        T_wall=T_wall_pw, k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
    )),
    rtol=RTOL, atol=ATOL, n_sec=N_SEC, show_progress=True,
)
print(f"✓  Ts={g_3d._Ts_results[-1,0]-273.15:.0f} °C | ρ_bio={g_3d._rho_solid_results[-1,0,0]:.2f} kg/m³")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("3D — Piecewise T_wall: secado → plateau → pirólisis  (0D, batch)", fontsize=11)

axes[0].plot(t_3d/60, [resolve(T_wall_pw, ti) for ti in t_3d], 'm-', lw=2)
axes[0].set(xlabel="t [min]", ylabel="T_wall [K]", title="Perfil piecewise"); axes[0].grid(True)

axes[1].plot(t_3d/60,  g_3d._Ts_results[:,0]-273.15, 'm-',  label='piecewise')
axes[1].plot(t_ref/60, g_ref._Ts_results[:,0]-273.15, 'k--', label='referencia')
axes[1].set(xlabel="t [min]", ylabel="Ts [°C]", title="Temperatura sólido")
axes[1].legend(); axes[1].grid(True)

axes[2].plot(t_3d/60, g_3d._rho_solid_results[:,0,0], 'm-',  label='biomasa')
axes[2].plot(t_3d/60, g_3d._rho_solid_results[:,2,0], 'm:',  alpha=0.7, label='humedad')
axes[2].plot(t_ref/60, g_ref._rho_solid_results[:,0,0], 'k--', label='bio ref')
axes[2].set(xlabel="t [min]", ylabel="ρ [kg/m³_bed]", title="Densidades sólidas")
axes[2].legend(fontsize=8); axes[2].grid(True)

axes[3].plot(t_3d/60,  g_3d._P_results[:,0], 'm-',  label='piecewise')
axes[3].plot(t_ref/60, g_ref._P_results[:,0], 'k--', label='referencia')
axes[3].set(xlabel="t [min]", ylabel="P [bar]", title="Presión")
axes[3].legend(); axes[3].grid(True)

plt.tight_layout(); plt.show()
print("\n--- Balances 3D ---")
tbc_3d = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=T_wall_pw, k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
)
check_balances(g_3d, make_params(tbc_3d))

## 3E — Control proporcional: T_wall ajustada para mantener Ts_mean ≈ 700 K

Un controlador P ajusta T_wall en **cada llamada al RHS** (incluida la estimación del
Jacobiano BDF) usando el snap interno del RHS — `Ts_mean` del sólido en ese instante.

**Controlador:** `T_wall = clamp(800 + 3×(700−Ts_mean), 300, 1200)` [K]  
**Qué observar:** Ts se regula cerca del SP; el controlador P tiene error en estado
estacionario (no hay integral). La pirólisis es más lenta que con T_wall=1073 K.

In [ ]:
SP_Ts = 700.0   # [K]  setpoint de Ts_mean

ctrl_P = proportional(
    setpoint=SP_Ts, gain=3.0,
    channel_in=lambda snap: snap.get("Ts_mean", 300.0),
    output_bias=800.0, output_min=300.0, output_max=1200.0,
)

tbc_3e = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=ctrl_P,           # callable(t, snap) → float
    k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
)

print(f"Simulando 3E (ctrl P T_wall, SP_Ts={SP_Ts} K)...")
t_3e, _, g_3e = run_step(
    sv0=sv0, t_max=T_END, params=make_params(tbc_3e),
    rtol=RTOL, atol=ATOL, n_sec=N_SEC, show_progress=True,
)
print(f"✓  Ts={g_3e._Ts_results[-1,0]-273.15:.0f} °C | ρ_bio={g_3e._rho_solid_results[-1,0,0]:.2f} kg/m³")

# Reconstruir T_wall efectiva post-hoc: T_wall = clamp(800 + 3*(700 - Ts), 300, 1200)
Ts_hist   = g_3e._Ts_results[:,0]
T_wall_eff = np.clip(800.0 + 3.0 * (SP_Ts - Ts_hist), 300.0, 1200.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle(f"3E — Control P: T_wall ajustada para Ts_mean = {SP_Ts} K  (0D, batch)", fontsize=11)

axes[0].plot(t_3e/60, T_wall_eff, color='darkorange', lw=2)
axes[0].axhline(800.0, color='k', ls='--', alpha=0.4, label='output_bias')
axes[0].set(xlabel="t [min]", ylabel="T_wall efectiva [K]", title="T_wall del ctrl P")
axes[0].legend(); axes[0].grid(True)

axes[1].plot(t_3e/60, Ts_hist-273.15, color='darkorange', label='ctrl P')
axes[1].axhline(SP_Ts-273.15, color='r', ls='--', label=f'SP={SP_Ts-273.15:.0f} °C')
axes[1].plot(t_ref/60, g_ref._Ts_results[:,0]-273.15, 'k--', alpha=0.4, label='referencia')
axes[1].set(xlabel="t [min]", ylabel="Ts [°C]", title="Temperatura sólido")
axes[1].legend(fontsize=8); axes[1].grid(True)

axes[2].plot(t_3e/60, g_3e._rho_solid_results[:,0,0], color='darkorange', label='ctrl P')
axes[2].plot(t_ref/60, g_ref._rho_solid_results[:,0,0], 'k--', alpha=0.4, label='referencia')
axes[2].set(xlabel="t [min]", ylabel="ρ_bio [kg/m³_bed]", title="Biomasa restante")
axes[2].legend(); axes[2].grid(True)

axes[3].plot(t_3e/60, g_3e._P_results[:,0], color='darkorange', label='ctrl P')
axes[3].plot(t_ref/60, g_ref._P_results[:,0], 'k--', alpha=0.4, label='referencia')
axes[3].set(xlabel="t [min]", ylabel="P [bar]", title="Presión")
axes[3].legend(); axes[3].grid(True)

plt.tight_layout(); plt.show()
print("\n--- Balances 3E ---")
check_balances(g_3e, make_params(tbc_3e))

## Resumen comparativo — todos los casos

In [ ]:
casos = [
    ("Ref — T_wall=1073 K cte",    t_ref, g_ref, 'k',          '--'),
    ("3A — ramp 300→1073 K",       t_3a,  g_3a,  'r',          '-'),
    ("3B — step 500→1073 K @600s", t_3b,  g_3b,  'b',          '-'),
    ("3C — pulse Qwall 500W/600s",  t_3c,  g_3c,  'g',          '-'),
    ("3D — piecewise 4 etapas",    t_3d,  g_3d,  'm',          '-'),
    ("3E — ctrl P Ts_SP=700 K",    t_3e,  g_3e,  'darkorange', '-'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Comparación — todos los casos (0D, batch, 3600 s)", fontsize=12)
for lbl, t_, g_, col, ls in casos:
    axes[0].plot(t_/60, g_._Ts_results[:,0]-273.15,    color=col, ls=ls, lw=1.5, label=lbl)
    axes[1].plot(t_/60, g_._rho_solid_results[:,0,0],  color=col, ls=ls, lw=1.5, label=lbl)
    axes[2].plot(t_/60, g_._P_results[:,0],            color=col, ls=ls, lw=1.5, label=lbl)
axes[0].set(xlabel="t [min]", ylabel="Ts [°C]",          title="Temperatura del sólido")
axes[1].set(xlabel="t [min]", ylabel="ρ_bio [kg/m³_bed]",title="Biomasa restante")
axes[2].set(xlabel="t [min]", ylabel="P [bar]",           title="Presión")
for ax in axes:
    ax.legend(fontsize=7); ax.grid(True)
plt.tight_layout(); plt.show()

# Tabla de métricas finales
rows = []
for lbl, t_, g_, *_ in casos:
    bio0 = g_._rho_solid_results[0,0,0]
    biof = g_._rho_solid_results[-1,0,0]
    rows.append({
        "caso":              lbl,
        "Ts_fin [°C]":       round(float(g_._Ts_results[-1,0])-273.15, 0),
        "P_max [bar]":       round(float(g_._P_results[:,0].max()), 3),
        "ρ_bio_fin [kg/m³]": round(float(biof), 2),
        "conv_bio [%]":      round(100*(1-biof/bio0) if bio0>0 else 0, 1),
    })
display(pd.DataFrame(rows).set_index("caso"))

## Conclusiones

**Qué demuestra este test:**
- Los tres tipos de señal BC (`float`, `callable(t)`, `callable(t, snap)`) funcionan
  sin cambiar nada en el RHS ni en el runner. Los balances cierran en todos los casos.
- El solver BDF maneja correctamente señales discontinuas (escalón, pulso).
- El snap del RHS alimenta correctamente al controlador P: `Ts_mean` se actualiza
  en cada llamada al RHS durante la integración.

**Diferencias entre casos:**

| Caso | Señal | Efecto principal |
|------|-------|------------------|
| Ref  | cte 1073 K | Conversión rápida desde t=0 |
| 3A ramp | 300→1073 K/3600s | Retardo de pirólisis; misma Ts final |
| 3B step | 500→1073 K @10min | Secado en fase 1; pirólisis brusca con el salto |
| 3C pulse | 500W×10min | Energía insuficiente → conversión parcial |
| 3D piecewise | 4 etapas | Velocidad de conversión controlada por tramos |
| 3E ctrl P | Ts_mean→SP=700K | Ts regulada; pirólisis más lenta que referencia |

**Próximos pasos:**
- Control `onoff` de T_wall y comparación con ctrl P.
- Señales en `v_gas_in` para simular arranque/corte del agente gasificante.
- Subir a 1D (N=10) para ver respuesta espacial ante señales temporales.
- `test_gasifier_04_0D_control.ipynb` — PID, barrido paramétrico, optimización.